# SDXL LoRA 数据品控 A/B
在 Colab 选择 GPU，执行全部单元。训练两组相同配置，再用相同 Prompt、Seed 与采样设置生成盲评对照。本页面默认跑全尺寸 SDXL，下载数 GB 的公开权重。

In [ ]:
from pathlib import Path
import subprocess, sys

repo = Path.cwd() if Path('curate_lora_dataset.py').is_file() else Path('lora-data-curation-exp').resolve()
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/lubin010319/lora-data-curation-exp.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-train.txt'], cwd=repo, check=True)
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime first.'
print(torch.cuda.get_device_name())


In [ ]:
def run(*args):
    subprocess.run([sys.executable, *args], cwd=repo, check=True)

run('curate_lora_dataset.py')
for group in ('a', 'b'):
    run('train_lora.py', f'configs/train_{group}.toml')
run('train_lora.py', 'configs/train_a.toml', 'configs/train_b.toml', '--evaluate')


In [ ]:
from IPython.display import Markdown, display
from IPython.display import Image as DisplayImage

for path in sorted((repo / 'work' / 'eval').glob('trial_*.png')):
    display(Markdown(path.stem), DisplayImage(filename=str(path), width=360))
display(Markdown('在 `work/eval/scorecard.json` 写入人工评分。定稿后打开 `key.json`。'))
